I want to check correlation of certian features with p-tau

# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('../')
from settings import *

# Preprocess

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'
df = pd.read_excel(DATA_PATH, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head(1)

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   'Spatial ': 'Spatial'
                   }, inplace=True)

In [ ]:
COGNITIVE_TESTS = [MCI, 'MOCA_score', 'Executive', 'Spatial', 'Attention_and_concentration', 'Memory', 'Language', 'Orientation', 'Frontal memory score (#1)', 'Hipocampal memory score (#2)', 'Trail_B_(second)', 'IST']
BEHAVIORAL_TESTS = ['Mild_behavioral_Impairment_Checklist', 'interest, motivation, and drive', 'mood or anxiety symptoms', 'Impulse control', 'following societal norms', 'strongly held beliefs and sensory', 'iADL']
RISK_FACTORS = ['height', 'Weight', 'Heart_disease', 'Diabetes', 'Hypertension', 'Hyperlipidemia',
                'Periodontal disease', 'Recurrent_UTI/_pneumonia/other_infection', 'Systemic_inflammatory_disease', 'Smoking', 'Pack_years',
                'Alcohol_consumption', 'Insomnia', 'OSA', 'Trearment_for_OSA', 'severity of OSA', 'Physical activity', 'Time_Physical_activity_(hours/week)',
                'Anxiety/depression','Cognitive_activity', 'AD_in_the_family', 'Number_of_risk_factors']

BIOMARKES = [TAU_217, TAU_CAT,'GFAP', 'NFL']

In [ ]:
df = df[df[MOCA_COL].notnull()]
df = df[df[MOCA_COL] != '?']

In [ ]:
df[TAU_CAT] = df[TAU_217] > HIGH_TAU_217_THR
df[MCI] = df[MOCA_COL].astype(int) <= MCI_THR

## Cognitive Tests

In [ ]:
df_cognitive = df[COGNITIVE_TESTS + ['Mild_behavioral_Impairment_Checklist'] + BIOMARKES]

In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(df_cognitive.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of cognitive tests and biomarkers')

plt.show()

## Behavioral Tests

In [ ]:
df_behavioral = df[BEHAVIORAL_TESTS + BIOMARKES]

In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(df_behavioral.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of behavioral tests and biomarkers')

plt.show()

## Risk Factors

In [ ]:
# changing smoking manually
df['Smoking'][df['Smoking'] == 'NO'] = 0
df['Smoking'][df['Smoking'] == 'in_the_past'] = 1
df['Smoking'][df['Smoking'] == 'YES'] = 2
df.Smoking = df.Smoking.astype(int)

In [ ]:
columns_to_encode = ['Cognitive_activity', 'Insomnia', 'Alcohol_consumption', 'Heart_disease', 'Diabetes', 'Hypertension', 'Hyperlipidemia', 'Periodontal disease', 'Recurrent_UTI/_pneumonia/other_infection', 'Systemic_inflammatory_disease', 'OSA', 'Trearment_for_OSA', 'Physical activity', 'Anxiety/depression', 'AD_in_the_family']

# Some horrible manual fixing
df['Heart_disease'][df['Heart_disease'] == 'NO '] = 'NO' # extra space
df['Trearment_for_OSA'][df['Trearment_for_OSA'] == 'YES CPAP'] = 'YES' # It's only one patient at the moment
df['Family_status'][df['Family_status'] == 'single '] = 'single' # extra space
df['Physical activity'][df['Physical activity'] == 'MEDIUM '] = 'MEDIUM' # extra space
df['Physical activity'][df['Physical activity'] == 'LOW '] = 'LOW' # extra space
df['Physical activity'][df['Physical activity'] == 'HIGH '] = 'HIGH'  # extra space
df['Ethnicity'][df['Ethnicity'] == 'Jewish'] = 'JEWISH' # Only capital letters


for col in columns_to_encode:
    unique_values = df[col].unique()
    if ('NO' in unique_values):
      df[col] = (~(df[col] == 'NO')).astype(int) # NO == 0, YES == 1
    elif len(unique_values) == 2:
        # If a column has exactly two unique values, binarize it
        df[col] = (df[col] == unique_values[0]).astype(int)
    else:
        # Otherwise, one-hot encode it
        df = pd.get_dummies(df, columns=[col])

    print(col, unique_values)

In [ ]:
df_rf = df[RISK_FACTORS + BIOMARKES]

In [ ]:
plt.figure(figsize=(20, 8))
sns.heatmap(df_rf.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of risk factors and biomarkers')

plt.show()

# Cognitive Healthy only

In [ ]:
print(df.shape, df[MCI].sum())
# df = df[~df[MCI]]
print(df.shape)

## Cognitive Tests

In [ ]:
df_cognitive = df[COGNITIVE_TESTS + ['Mild_behavioral_Impairment_Checklist'] + BIOMARKES]

In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(df_cognitive.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of cognitive tests and biomarkers')

plt.show()

## Behavioral Tests

In [ ]:
df_behavioral = df[BEHAVIORAL_TESTS + BIOMARKES]

In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(df_behavioral.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of behavioral tests and biomarkers')

plt.show()

## Risk Factors

In [ ]:
# changing smoking manually
df['Smoking'][df['Smoking'] == 'NO'] = 0
df['Smoking'][df['Smoking'] == 'in_the_past'] = 1
df['Smoking'][df['Smoking'] == 'YES'] = 2
df.Smoking = df.Smoking.astype(int)

In [ ]:
columns_to_encode = ['Cognitive_activity', 'Insomnia', 'Alcohol_consumption', 'Heart_disease', 'Diabetes', 'Hypertension', 'Hyperlipidemia', 'Periodontal disease', 'Recurrent_UTI/_pneumonia/other_infection', 'Systemic_inflammatory_disease', 'OSA', 'Trearment_for_OSA', 'Physical activity', 'Anxiety/depression', 'AD_in_the_family']

# Some horrible manual fixing
df['Heart_disease'][df['Heart_disease'] == 'NO '] = 'NO' # extra space
df['Trearment_for_OSA'][df['Trearment_for_OSA'] == 'YES CPAP'] = 'YES' # It's only one patient at the moment
df['Family_status'][df['Family_status'] == 'single '] = 'single' # extra space
df['Physical activity'][df['Physical activity'] == 'MEDIUM '] = 'MEDIUM' # extra space
df['Physical activity'][df['Physical activity'] == 'LOW '] = 'LOW' # extra space
df['Physical activity'][df['Physical activity'] == 'HIGH '] = 'HIGH'  # extra space
df['Ethnicity'][df['Ethnicity'] == 'Jewish'] = 'JEWISH' # Only capital letters


for col in columns_to_encode:
    unique_values = df[col].unique()
    if ('NO' in unique_values):
      df[col] = (~(df[col] == 'NO')).astype(int) # NO == 0, YES == 1
    elif len(unique_values) == 2:
        # If a column has exactly two unique values, binarize it
        df[col] = (df[col] == unique_values[0]).astype(int)
    else:
        # Otherwise, one-hot encode it
        df = pd.get_dummies(df, columns=[col])

    print(col, unique_values)

In [ ]:
df_rf = df[RISK_FACTORS + BIOMARKES]

In [ ]:
plt.figure(figsize=(20, 8))
sns.heatmap(df_rf.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of risk factors and biomarkers')

plt.show()

# AD MCI vs non-AD MCI

In [ ]:
print(df.shape)
df = df[df[MCI]]
print(df.shape)

## Cognitive Tests

In [ ]:
df_cognitive = df[COGNITIVE_TESTS + ['Mild_behavioral_Impairment_Checklist'] + BIOMARKES]
df_cognitive.drop(columns=['MCI'], inplace=True)

In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(df_cognitive.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of cognitive tests and biomarkers')

plt.show()

## Behavioral Tests

In [ ]:
df_behavioral = df[BEHAVIORAL_TESTS + BIOMARKES]

In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(df_behavioral.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of behavioral tests and biomarkers')

plt.show()

## Risk Factors

In [ ]:
# changing smoking manually
df['Smoking'][df['Smoking'] == 'NO'] = 0
df['Smoking'][df['Smoking'] == 'in_the_past'] = 1
df['Smoking'][df['Smoking'] == 'YES'] = 2
df.Smoking = df.Smoking.astype(int)

In [ ]:
columns_to_encode = ['Cognitive_activity', 'Insomnia', 'Alcohol_consumption', 'Heart_disease', 'Diabetes', 'Hypertension', 'Hyperlipidemia', 'Periodontal disease', 'Recurrent_UTI/_pneumonia/other_infection', 'Systemic_inflammatory_disease', 'OSA', 'Trearment_for_OSA', 'Physical activity', 'Anxiety/depression', 'AD_in_the_family']

# Some horrible manual fixing
df['Heart_disease'][df['Heart_disease'] == 'NO '] = 'NO' # extra space
df['Trearment_for_OSA'][df['Trearment_for_OSA'] == 'YES CPAP'] = 'YES' # It's only one patient at the moment
df['Family_status'][df['Family_status'] == 'single '] = 'single' # extra space
df['Physical activity'][df['Physical activity'] == 'MEDIUM '] = 'MEDIUM' # extra space
df['Physical activity'][df['Physical activity'] == 'LOW '] = 'LOW' # extra space
df['Physical activity'][df['Physical activity'] == 'HIGH '] = 'HIGH'  # extra space
df['Ethnicity'][df['Ethnicity'] == 'Jewish'] = 'JEWISH' # Only capital letters


for col in columns_to_encode:
    unique_values = df[col].unique()
    if ('NO' in unique_values):
      df[col] = (~(df[col] == 'NO')).astype(int) # NO == 0, YES == 1
    elif len(unique_values) == 2:
        # If a column has exactly two unique values, binarize it
        df[col] = (df[col] == unique_values[0]).astype(int)
    else:
        # Otherwise, one-hot encode it
        df = pd.get_dummies(df, columns=[col])

    print(col, unique_values)

In [ ]:
df_rf = df[RISK_FACTORS + BIOMARKES]

In [ ]:
plt.figure(figsize=(20, 8))
sns.heatmap(df_rf.corr(), annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Correlation heatmap of risk factors and biomarkers')

plt.show()